[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/01_measurement.ipynb)

# Paper 1: the determinacy ceiling, identification, and sample size

Three questions. How good can a factor score for this construct ever be? Is the model
identified? And how many people does fitting it need? The first has a closed-form answer
that no amount of item writing can move, and that is the result worth your time.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 30 seconds. Full run: about three minutes, nearly all of it the sample-size simulation.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here takes about three minutes.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## The model

Indicators load on domains, domains load on one general construct:

$$y = \Lambda\eta + \varepsilon, \qquad \eta = \gamma\zeta + \delta,$$

with $\operatorname{Var}(\zeta) = 1$, $\operatorname{Var}(\delta) = \Psi$ and
$\operatorname{Var}(\varepsilon) = \Theta$. Writing $\Phi = \gamma\gamma^{\top} + \Psi$,
the model implies $\Sigma = \Lambda\Phi\Lambda^{\top} + \Theta$ and
$\operatorname{cov}(y,\zeta) = \Lambda\gamma$.

The regression (Thurstone) score has determinacy

$$\rho_D^2 = (\Lambda\gamma)^{\top}\Sigma^{-1}(\Lambda\gamma).$$

This is a population quantity. It does not depend on the sample size. Collecting more
people sharpens the estimates of $\Lambda$, $\gamma$ and $\Psi$; it does not make any one
person's score more determinate.

## Deriving the ceiling

Indicators reach $\zeta$ only through $\eta$, so $\zeta \to \eta \to y$ is a Markov chain.
Everything is jointly Gaussian, so the score is $\widehat\zeta = E(\zeta \mid y)$ and
$\rho_D^2 = \operatorname{Var}\{E(\zeta \mid y)\}$. The tower property gives
$E(\zeta \mid y) = E\{E(\zeta \mid \eta) \mid y\}$, and the law of total variance gives

$$\operatorname{Var}\{E(\zeta\mid\eta)\}
 = \operatorname{Var}\{E(\zeta\mid y)\}
 + E\!\left[\operatorname{Var}\{E(\zeta\mid\eta)\mid y\}\right].$$

The last term is strictly positive whenever $\Theta \succ 0$, so no score built from the
indicators beats a score built from the domain factors themselves. And with
$\eta = \gamma\zeta + \delta$,

$$\operatorname{Var}\{E(\zeta\mid\eta)\} = \gamma^{\top}(\gamma\gamma^{\top}+\Psi)^{-1}\gamma
 = c - \frac{c^{2}}{1+c} = \frac{c}{1+c},
 \qquad c = \gamma^{\top}\Psi^{-1}\gamma,$$

by Sherman-Morrison. Hence

$$\rho_D < \rho_D^{\infty} = \sqrt{\frac{c}{1+c}},$$

approached from below as indicators per domain grow. A target $\rho_D \ge r$ needs
$c \ge r^2/(1-r^2)$, so $r = 0.90$ needs $c \ge 4.2632$.

In [ ]:
banner()

from hpotential.core.measurement import (
    determinacy_ceiling, determinacy_curve, required_second_order_signal, second_order_determinacy,
)

GAMMA = np.array([0.80, 0.70, 0.62])   # second-order loadings of the planned design
PSI = np.array([0.36, 0.51, 0.62])     # domain disturbance variances

rho_inf, c = determinacy_ceiling(GAMMA, PSI)
needed = required_second_order_signal(0.90)

print(f"c = gamma' Psi^-1 gamma          {c:.6f}")
print(f"ceiling  sqrt(c / (1 + c))       {rho_inf:.6f}")
print(f"c needed to reach rho_D = 0.90   {needed:.4f}")
print(f"the planned design is short by   {needed - c:.4f}")

### Check the closed form against a direct evaluation

The ceiling is a limit. Evaluate $\rho_D$ the long way, from the full model-implied
covariance at 400 indicators per domain, and it should sit just under the ceiling. Note
which side it falls on: the theorem says strictly below, always.

In [ ]:
banner()

direct = second_order_determinacy(400, 0.85, GAMMA, PSI)
print(f"closed-form ceiling                        {rho_inf:.6f}")
print(f"direct evaluation, 400 indicators/domain   {direct:.6f}")
print(f"gap                                        {rho_inf - direct:.3e}")
print(f"strictly below the ceiling, as the theorem requires: {direct < rho_inf}")
print()
print("1200 indicators. Three decimal places short of a number that is itself")
print("short of the 0.90 an individual-level decision would want.")

### The curve, with the ceiling drawn on it

Four standardised loadings, indicators per domain on a log axis. The dotted line is the
ceiling and the dashed line is 0.90.

In [ ]:
banner()

ms = np.arange(3, 41)
curve = determinacy_curve(ms, (0.55, 0.65, 0.75, 0.85), GAMMA, PSI)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for i, col in enumerate(curve.columns):
    ax.plot(curve.index, curve[col], lw=1.5, color=series_color(i), label=col.replace("lambda=", r"$\lambda$ = "))
ax.axhline(rho_inf, ls=":", lw=1.4, color=series_color(5))
ax.axhline(0.90, ls="--", lw=1.0, color=INK)
ax.text(40, rho_inf + 0.004, rf"ceiling $\rho_D^\infty$ = {rho_inf:.4f}", ha="right", fontsize=8,
        color=series_color(5))
ax.text(40, 0.904, "individual-use benchmark 0.90", ha="right", fontsize=8, color=INK)
ax.set_xscale("log")
ax.set_xticks([3, 4, 6, 8, 12, 20, 40])
ax.set_xticklabels(["3", "4", "6", "8", "12", "20", "40"])
ax.set_ylim(0.70, 0.95)
ax.set_xlabel("indicators per domain")
ax.set_ylabel(r"factor-score determinacy $\rho_D$")
ax.set_title("Every curve flattens under the same line", fontsize=10)
ax.legend(fontsize=8, loc="lower right", ncol=2)
plt.show()

print("Raising the loading lifts the curve. It does not lift the line.")

## The interactive part: move $\gamma$ and $\Psi$, watch the ceiling

This is the moment the paper is built around. The sliders below set the three second-order
loadings and the three domain disturbances. Watch two numbers: the ceiling, and how many
indicators per domain you would need to reach 0.90.

Try to reach 0.90 by adding indicators. You cannot. The only way is up the $\gamma$
sliders or down the $\Psi$ ones, because those are the only things $c$ depends on.

For speed the search over indicator counts uses a $K \times K$ reduction of the same
algebra rather than inverting a large covariance matrix. The next cell checks that the
reduction agrees with the library function before anything is built on it.

In [ ]:
banner()


def rho_reduced(m, lam, gamma, psi):
    """rho_D for a balanced design, computed on K x K matrices.

    Woodbury on Sigma = Lambda Phi Lambda' + Theta turns the p x p inverse into a
    K x K one, with W = Lambda' Theta^-1 Lambda = diag(m lam^2 / theta_k):

        rho_D^2 = gamma' W gamma - gamma' W (Phi^-1 + W)^-1 W gamma.

    Same quantity, cheap enough to sit behind a slider.
    """
    gamma = np.asarray(gamma, float)
    psi = np.asarray(psi, float)
    theta = np.clip(1.0 - lam**2 * (gamma**2 + psi), 1e-3, None)
    w = np.diag(m * lam**2 / theta)
    phi = np.outer(gamma, gamma) + np.diag(psi)
    inner = np.linalg.inv(np.linalg.inv(phi) + w)
    q = gamma @ w @ gamma - gamma @ w @ inner @ w @ gamma
    return float(np.sqrt(np.clip(q, 0.0, 1.0)))


print(f"{'m':>6} {'library':>12} {'reduction':>12} {'difference':>12}")
for m in (1, 4, 8, 40, 400):
    a = second_order_determinacy(m, 0.85, GAMMA, PSI)
    b = rho_reduced(m, 0.85, GAMMA, PSI)
    print(f"{m:>6} {a:>12.9f} {b:>12.9f} {abs(a - b):>12.2e}")

In [ ]:
banner()


def indicators_for_target(lam, gamma, psi, target=0.90, m_max=100_000):
    """Smallest indicators per domain reaching the target, or None if unreachable."""
    ceiling, _ = determinacy_ceiling(np.asarray(gamma), np.asarray(psi))
    if ceiling <= target:
        return None
    lo, hi = 1, 2
    while rho_reduced(hi, lam, gamma, psi) < target:
        hi *= 2
        if hi > m_max:
            return None
    while lo < hi:
        mid = (lo + hi) // 2
        if rho_reduced(mid, lam, gamma, psi) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


def ceiling_panel(g1=0.80, g2=0.70, g3=0.62, p1=0.36, p2=0.51, p3=0.62, lam=0.75):
    gamma = np.array([g1, g2, g3])
    psi = np.array([p1, p2, p3])
    ceiling, c_now = determinacy_ceiling(gamma, psi)
    m_needed = indicators_for_target(lam, gamma, psi)

    ms = np.unique(np.round(np.logspace(0, 3, 60)).astype(int))
    rho = [rho_reduced(int(m), lam, gamma, psi) for m in ms]

    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    ax.plot(ms, rho, lw=1.6, color=series_color(0), label=rf"$\lambda$ = {lam:.2f}")
    ax.axhline(ceiling, ls=":", lw=1.4, color=series_color(5))
    ax.axhline(0.90, ls="--", lw=1.0, color=INK)
    ax.set_xscale("log")
    ax.set_ylim(0.4, 1.0)
    ax.set_xlabel("indicators per domain (log scale)")
    ax.set_ylabel(r"$\rho_D$")
    ax.legend(fontsize=8, loc="lower right")
    reach = "unreachable at any number of indicators" if m_needed is None else f"{m_needed} per domain"
    ax.set_title(f"c = {c_now:.3f},  ceiling = {ceiling:.4f},  0.90 needs {reach}", fontsize=9)
    plt.show()

    print(f"second-order signal c        {c_now:.4f}   (0.90 needs c >= 4.2632)")
    print(f"ceiling                      {ceiling:.4f}")
    print(f"indicators per domain for 0.90: {reach}")


explore(
    ceiling_panel,
    controls=dict(
        g1=(0.10, 0.99, 0.01), g2=(0.10, 0.99, 0.01), g3=(0.10, 0.99, 0.01),
        p1=(0.05, 0.95, 0.01), p2=(0.05, 0.95, 0.01), p3=(0.05, 0.95, 0.01),
        lam=(0.30, 0.95, 0.05),
    ),
    defaults=dict(g1=0.80, g2=0.70, g3=0.62, p1=0.36, p2=0.51, p3=0.62, lam=0.75),
)

**What the sliders show.** The horizontal ceiling moves only when $\gamma$ or $\Psi$
moves. The curve slides up and down with the loading, and stretches right forever, and
still never crosses the ceiling. When the ceiling is below 0.90 the answer to "how many
indicators do I need" is not a large number. There is no number.

**What they do not show.** Whether any real instrument has these $\gamma$ and $\Psi$. The
ceiling is computed from a hypothesised second-order structure. If the true loadings are
larger, the ceiling is higher, and the result still holds with the new $c$.

## Identification

A model is locally identified at a parameter point when the Jacobian of the map from free
parameters to unique model-implied moments has full column rank there (Rothenberg, 1971).
With 12 indicators there are $12 \cdot 13 / 2 = 78$ unique moments and, under marker
scaling, 27 free parameters.

A rank check that never fails is not a check. So the run below includes two controls. The
first, two indicators per domain over three domains, is the case people expect to fail; it
does not, because the second-order factor supplies the missing restrictions. The second, a
second-order factor over only two domains, must fail: two loadings and two disturbances
cannot be recovered from a single between-domain covariance.

In [ ]:
banner()

from hpotential.core.measurement import identification_report

reports = [identification_report(lam, f"planned design, loading {lam}", GAMMA, PSI, per_domain=4)
           for lam in (0.55, 0.62, 0.75, 0.90)]
reports.append(identification_report(0.75, "control (a): 2 indicators per domain, 3 domains",
                                     GAMMA, PSI, per_domain=2))
reports.append(identification_report(0.75, "control (b): second-order factor over 2 domains",
                                     GAMMA[:2], PSI[:2], per_domain=4))

table = pd.DataFrame(reports)[
    ["label", "n_free", "n_moments", "rank", "identified", "smallest_singular_value", "condition_number"]
]
table.columns = ["design", "free", "moments", "rank", "identified", "smallest sv", "condition"]
table

In [ ]:
banner()

ctrl = reports[-1]
print("The negative control:")
print(f"  {ctrl['label']}")
print(f"  rank {ctrl['rank']} of {ctrl['n_free']} free parameters -> identified: {ctrl['identified']}")
print(f"  smallest singular value {ctrl['smallest_singular_value']:.3e}")
print(f"  condition number        {ctrl['condition_number']:.3e}")
print()
print("A smallest singular value at 1e-11 next to condition numbers around 11 for the")
print("planned designs is the difference between a rank deficiency and rounding.")

## Sample size

Determinacy does not depend on $n$, but everything else does: whether the fit converges,
whether the solution is admissible, whether the estimates are unbiased, and whether a
nominal 95 per cent interval covers 95 per cent of the time. Coverage is counted here, not
assumed. A study that reports 0.95 without counting has reported the nominal level.

The run below is a simulation. Under a reduced run it uses a short grid and few replicates,
so its numbers are noisy and are not the published ones.

In [ ]:
banner()

import time

from hpotential.analyses import paper1_validation
from hpotential.config import Paper1Config

if FULL_RUN:
    cfg1 = Paper1Config()                                     # 150 reps, six sample sizes
else:
    cfg1 = Paper1Config(n_grid=(150, 400, 900), reps=10)      # noisy on purpose, and quick

start = time.time()
res1 = paper1_validation.run(cfg1)
print(f"{cfg1.reps} replicates per point, {len(cfg1.n_grid)} sample sizes, "
      f"{time.time() - start:.1f} s")

summary = res1["sample_size"]["summary"]
rows = [{"scenario": v["scenario"], "n": v["n"], "converged": v["convergence"],
         "admissible": v["admissible"], "worst coverage": v["min_coverage"],
         "max |rel bias|": v["max_abs_rel_bias"], "rho_D": v["determinacy"]}
        for v in summary.values()]
pd.DataFrame(rows).round(4)

In [ ]:
banner()

frame = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.2))
for i, scenario in enumerate(sorted(frame["scenario"].unique())):
    part = frame[frame["scenario"] == scenario].sort_values("n")
    axes[0].plot(part["n"], part["admissible"], "o-", ms=4, lw=1.3, color=series_color(i),
                 label=scenario)
    axes[1].plot(part["n"], part["worst coverage"], "o-", ms=4, lw=1.3, color=series_color(i))
    axes[2].plot(part["n"], part["rho_D"], "o-", ms=4, lw=1.3, color=series_color(i))
axes[0].axhline(0.95, ls="--", lw=0.8, color=INK); axes[0].set_ylabel("admissible solutions")
axes[1].axhline(0.95, ls="--", lw=0.8, color=INK); axes[1].set_ylabel("worst 95% coverage")
axes[2].axhline(0.90, ls="--", lw=0.8, color=INK); axes[2].set_ylabel(r"$\rho_D$")
axes[2].axhline(rho_inf, ls=":", lw=1.1, color=series_color(5))
for ax in axes:
    ax.set_xscale("log"); ax.set_xlabel("sample size $n$")
    ax.set_xticks(sorted(frame["n"].unique()))
    ax.set_xticklabels([str(n) for n in sorted(frame["n"].unique())], fontsize=7)
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

print("The third panel is the point: determinacy is flat in n and sits under the ceiling.")
print("More people buy better parameter estimates. They do not buy a better score.")

---

## What this does not establish

**The ceiling is algebra on an assumed population model.** It says nothing about whether
that model fits anything, and a determinacy computed from estimated parameters carries
their error. If the real $\gamma$ is larger than 0.80, 0.70, 0.62, the ceiling is higher,
and the theorem still holds at the new $c$.

**The result is about the regression score.** Bartlett scores have a different, generally
lower, determinacy. Swapping scoring methods does not escape the bound.

**Identification is local, not global.** A full-rank Jacobian at one parameter point rules
out a flat direction there. It does not rule out a distant equivalent solution.

**The sample-size run is a simulation of the planned model.** No participant has been
recruited. Coverage near 0.95 says the estimator behaves when the model is true, which is
the easy case.

**A reduced run does not reproduce the published values.** The closed-form results are
unaffected by the flag, because they involve no simulation. The sample-size numbers are.